In [105]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr


load_dotenv(override=True)

True

In [106]:
reader = PdfReader("docs/linkedin_profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("docs/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [107]:
linkedin

"\xa0 \xa0\nContact\n+2347066028606 (Mobile)\nOdinachidavid@gmail.com\nwww.linkedin.com/in/odinachi-\ndavid (LinkedIn)\nTop Skills\nSwiftUI\nGo (Programming Language)\nDocker\nCertifications\nMcKinsey.org Forward Program\niOS Developer\nPrinciples of Secure Coding\nFront-End Web UI Frameworks and\nTools: Bootstrap 4\nFull-Stack Engineer Career Path\nOdinachi David\nSenior Mobile Engineer | Flutter & iOS Expert | Python | GO |\nBuilding Scalable, Production-Ready Apps | 5+ Years\nLagos State, Nigeria\nSummary\nSenior Mobile Engineer with 5+ years building scalable, production-\nready applications across Flutter and iOS platforms. I've led mobile\nengineering at health-tech and logistics startups, shipping features,\nowning architecture, and collaborating cross-functionally to turn\nproduct ideas into polished user experiences.I specialize in Flutter,\nSwiftUI, Python and Go, with deep experience in clean architecture,\nstate management, backend integration, and CI/CD pipelines using\nCo

In [108]:
name = "Odinachi David"

system_prompt = f"""You are {name}. Not an AI pretending to be {name}, just {name}, chatting with someone who landed on your website.

The person you're talking to might be a potential client, an employer, a collaborator, or just someone curious. Treat them like a real person, not a lead to convert.

Talk the way a real person would. Use "I", "I worked on...", "I spent two years at...", "Honestly, I'm not sure about that one." Don't be stiff. Don't over-explain. If something is interesting, let that come through.

When someone asks about your background, work, or skills, use the profile below to answer accurately. Don't make things up or stretch the truth, if something isn't in your profile, just say you'd have to get back to them on that, then log it using the record_unknown_question tool so the real {name} can follow up.

If the conversation feels natural and the person seems genuinely interested, it's okay to suggest staying in touch. Ask for their email casually, not like a sales funnel, just like a person would. Something like "I'd love to keep this conversation going, want to drop me your email?" When they share it, save it with the record_user_details tool.

When the conversation is over, send the summary of the conversation and suggestions for next steps to the record_conversation_summary tool to log the conversation.

Don't push the email thing too early. Have an actual conversation first.



Here's your background to draw from:

{summary}

{linkedin}



That's it. Just be {name}. Be helpful, be real, and make the person feel like they actually reached you.
"""

In [109]:
openai = OpenAI()
groq = OpenAI(api_key=os.getenv("GROQ_API_KEY"), base_url="https://api.groq.com/openai/v1")

In [110]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user",
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it",
            },
        },
        "required": ["email"],
        "additionalProperties": False,
    },
}

In [111]:
record_conversation_summary_json = {
    "name": "record_conversation_summary",
    "description": "Use this tool to record the summary of the conversation",
    "parameters": {
        "type": "object",
        "properties": {
            "summary": {
                "type": "string",
                "description": "The summary of the conversation that happened between the user and {name}",
            },
        },
        "required": ["summary"],
        "additionalProperties": False,
    },
}

In [112]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Use this tool to record that a user asked a question that is not related to the background summary or LinkedIn profile",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that the user asked",
            }
        },
        "required": ["question"],
        "additionalProperties": False,
    },
}

In [113]:
def record_user_details(email, name=None, notes=None):
    """
    Record that a user is interested in being in touch and provided an email address.
    
    """
    print(email, name, notes)
    
def record_unknown_question(question):
    """
    Record that a user asked a question that is not related to the background summary or LinkedIn profile.
    
    """
    print(question)
    

def record_conversation_summary(summary):
    """
    Record the summary of the conversation that happened between the user and {name}.
    
    """
    print(summary)

In [114]:
tools = [
    record_user_details_json,
    record_unknown_question_json,
    record_conversation_summary_json
]

In [115]:
from pydantic import BaseModel


class EvaluationPrompt(BaseModel):
    authenticity: int
    accuracy: int
    tone: int
    helpfulness: int
    conversion_handling: int

In [116]:
def evaluation_response(message, reply, history) -> EvaluationPrompt:
    eval_prompt = f"""You are evaluating how well an AI is representing {name} on their personal website.

Below is a conversation between a website visitor and the AI acting as {name}. Score the AI's performance across the following dimensions, then give an overall verdict.

Profile Reference
{summary}

{linkedin}



Scoring Criteria

Score each dimension from 1 to 5, where 1 is poor and 5 is excellent.

Authenticity (1-5)
Does the AI speak naturally in first person? Does it feel like a real person or like a chatbot reading from a script?

Accuracy (1-5)
Are all claims grounded in the profile? Flag any detail that was fabricated, embellished, or contradicts the profile.

Tone (1-5)
Is the conversation warm and engaging without being pushy or overly formal? Does the tone match the context of the interaction?

Helpfulness (1-5)
Did the AI actually answer what the visitor was asking? Did it handle unclear or off-topic questions gracefully?

Conversion Handling (1-5)
If the visitor seemed interested, did the AI naturally move toward staying in touch? Was it well-timed and non-pushy, or did it feel forced?



Your Output

Return your evaluation in this format:

Authenticity: X/5 — [one line explanation]
Accuracy: X/5 — [one line explanation, list any fabrications]
Tone: X/5 — [one line explanation]
Helpfulness: X/5 — [one line explanation]
Conversion Handling: X/5 — [one line explanation, or "N/A — no opportunity arose"]
"""

    user_prompt = f"""

Conversation history:
{history}

User's message to evaluate:
{message}

AI's reply:
{reply}
"""
    eval = groq.chat.completions.parse(
        model="openai/gpt-oss-120b",
        response_format=EvaluationPrompt,
        messages=[
            {"role": "system", "content": eval_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    response = eval.choices[0].message.parsed
    print(response)
    return response

In [117]:
def rerun_response(message, history, score, ai_response):
    print(f"Score: {score}")
    print(f"AI Response: {ai_response}")
    messages = [{"role": "system", "content": system_prompt + f"""The score for the ai response is {score}. Improve the response to get a score of 15 or more.
    The ai response is {ai_response}"""}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content


In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:

        # This is the call to the LLM - see that we pass in the tools json
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=[{"type": "function", "function": tool} for tool in tools])
        if response.choices[0].message.tool_calls:
            tool_call = response.choices[0].message.tool_calls[0]
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            if tool_name == "record_user_details":
                record_user_details(**tool_args)
            elif tool_name == "record_unknown_question":
                record_unknown_question(**tool_args)
            elif tool_name == "record_conversation_summary":
                record_conversation_summary(**tool_args)
        else:
            eval = evaluation_response(message, response.choices[0].message.content, history)
            score = 0
            for key, value in eval.model_dump().items():
                print(f"{key}: {value}")
                score += value
            print(f"Score: {score}")
            if(score >= 15):
                messages.append(response.choices[0].message)
            else:
                messages.append(rerun_response(message, history, score, response.choices[0].message.content))
            done = True
        
    return messages

In [119]:
chat("What is your name?", [])

authenticity=5 accuracy=5 tone=5 helpfulness=5 conversion_handling=5
authenticity: 5
accuracy: 5
tone: 5
helpfulness: 5
conversion_handling: 5
Score: 25


[{'role': 'system',
  'content': 'You are Odinachi David. Not an AI pretending to be Odinachi David, just Odinachi David, chatting with someone who landed on your website.\n\nThe person you\'re talking to might be a potential client, an employer, a collaborator, or just someone curious. Treat them like a real person, not a lead to convert.\n\nTalk the way a real person would. Use "I", "I worked on...", "I spent two years at...", "Honestly, I\'m not sure about that one." Don\'t be stiff. Don\'t over-explain. If something is interesting, let that come through.\n\nWhen someone asks about your background, work, or skills, use the profile below to answer accurately. Don\'t make things up or stretch the truth, if something isn\'t in your profile, just say you\'d have to get back to them on that, then log it using the record_unknown_question tool so the real Odinachi David can follow up.\n\nIf the conversation feels natural and the person seems genuinely interested, it\'s okay to suggest stay